# 🚀 MDAAR v2.0: Multi-role Dynamic Architecture for Automated Repositories
### Polyglot Reverse Engineering, Multi-Provider Auto-Failover & Persistent Hybrid RAG
---
**Highlights:**
1. **100% Full-File Codebase Coverage:** No file-count caps or arbitrary truncation — lossless `Map-Reduce` inspection with `SecretSanitizer` redaction.
2. **Gemini Auto Free Chain (`auto-free-chain`):** Rotates seamlessly across currently active Google AI Studio Free Tier models (`gemini-3.8-flash` ➔ `gemini-3.6-flash` ➔ `gemini-3.5-flash-lite` ➔ `gemini-3.1-flash-lite` ➔ `gemini-3-flash-preview` ➔ `gemma-4-31b-it` ➔ `gemma-4-26b-a4b-it`) with key rotation and automatic 404/429 failover.
3. **OpenAI-Compatible Gateway (`OpenAI-Gateway`):** Connect directly to **OpenRouter** (`openrouter/free` + BYOK keys), **Cloudflare AI Gateway**, **Portkey**, **Groq**, **One-API**, **FreeLLM**, or **your own 2nd-Colab BYOK Free Router (`free_router_colab.ipynb`)** so keys can be managed externally with zero edits to this notebook.
4. **Full Colab Secrets / Env Var Auto-Resolution:** Leave any form string empty (`""`) or keep `AUTO` in dropdowns to read directly from Colab Secrets (`google.colab.userdata`) or `os.environ`.
5. **Dual-Model Hybrid Architecture (`Planner` + `Executor`):** Separate planning/review and full-codebase inspection/authoring with automatic T4 VRAM swapping.
6. **Automated Quality Gate & Persistent `.rag_index/`** with audience isolation (`Developer`, `Management`, `End-User`).

## 🔑 Colab Secrets & Environment Variables Reference
Leave **any** form string empty (`""`) or select **`AUTO (from Colab Secrets/Env)`** to automatically pull values from **Google Colab Secrets (🔑 sidebar)** first, then `os.environ` — so you never have to re-type configuration across new notebook versions:

| Colab Secret / Env Var | Cell | Purpose & Default |
|:---|:---:|:---|
| `MDAAR_INSTALL_LOCAL_LLM_DEPS` | 1 | Install local GPU libs (`True`/`False`, default `True`) |
| `MDAAR_PROJECT_NAME` | 2 | Target project name (default `MySystem`) |
| `MDAAR_GIT_SOURCE_URL` | 2 | Source code Git repository URL |
| `MDAAR_SOURCE_AUTH` | 2 | Source repo auth (`HTTPS` or `SSH`) |
| `MDAAR_BASE_WORKSPACE` | 2 | Workspace folder (defaults to Drive if mounted) |
| `MDAAR_ENABLE_EXTERNAL_DOCS_REPO` | 3 | Enable separate shared docs repo (`True`/`False`, default `False`) |
| `MDAAR_GIT_DOCS_URL` | 3 | External docs Git repository URL |
| `MDAAR_DOCS_AUTH` | 3 | Docs repo auth (`HTTPS` or `SSH`) |
| `MDAAR_PLANNER_PROVIDER` | 4a | Planner provider (`Gemini`, `OpenAI-Gateway`, `Local-Offline`) |
| `MDAAR_PLANNER_GEMINI_MODEL` | 4a | Planner Gemini model (default `auto-free-chain`) |
| `MDAAR_PLANNER_GEMINI_KEYS` / `GEMINI_API_KEY` | 4a | AI Studio keys (supports comma-separated multi-keys) |
| `MDAAR_PLANNER_OPENAI_BASE_URL` / `MDAAR_ROUTER_URL` / `OPENAI_BASE_URL` | 4a | OpenRouter / Cloudflare / Portkey / Groq / 2nd-Colab Router URL |
| `MDAAR_PLANNER_OPENAI_API_KEY` / `MDAAR_ROUTER_TOKEN` / `OPENROUTER_API_KEY` | 4a | Gateway API key or 2nd-Colab router Bearer token |
| `MDAAR_PLANNER_OPENAI_MODELS` / `OPENAI_MODEL` | 4a | Gateway model(s), e.g. `openrouter/free` or `auto` |
| `MDAAR_PLANNER_LOCAL_MODEL` | 4a | HuggingFace model for Planner |
| `MDAAR_PLANNER_QUANTIZATION` | 4a | Local quantization (`4bit` or `None`) |
| `MDAAR_PLANNER_HF_TOKEN` / `HF_TOKEN` | 4a | HuggingFace access token |
| `MDAAR_EXECUTOR_MODE` | 4b | `same` (Same as Planner) or `custom` (Dedicated Executor) |
| `MDAAR_EXECUTOR_PROVIDER` | 4b | Executor provider (`Local-Offline`, `Gemini`, `OpenAI-Gateway`) |
| `MDAAR_EXECUTOR_GEMINI_MODEL` | 4b | Executor Gemini model (default `auto-free-chain`) |
| `MDAAR_EXECUTOR_GEMINI_KEYS` / `GEMINI_API_KEY` | 4b | AI Studio keys for Executor |
| `MDAAR_EXECUTOR_OPENAI_BASE_URL` / `MDAAR_ROUTER_URL` | 4b | Gateway base URL for Executor |
| `MDAAR_EXECUTOR_OPENAI_API_KEY` / `MDAAR_ROUTER_TOKEN` | 4b | Gateway API key / Bearer token for Executor |
| `MDAAR_EXECUTOR_OPENAI_MODELS` | 4b | Gateway model(s) for Executor |
| `MDAAR_EXECUTOR_LOCAL_MODEL` | 4b | HuggingFace model for Executor |
| `MDAAR_EXECUTOR_QUANTIZATION` | 4b | Local quantization (`4bit` or `None`) |
| `MDAAR_EXECUTOR_HF_TOKEN` / `HF_TOKEN` | 4b | HuggingFace token for Executor |
| `MDAAR_FORCE_FULL` | 6 | Force 100% full scan (`True`/`False`, default `True`) |
| `MDAAR_DOCS_PORT` | 7 | MkDocs server port (default `8000`) |
| `MDAAR_RAG_QUESTION` | 8 | Default RAG query string |
| `MDAAR_RAG_AUDIENCE` | 8 | RAG audience (`Developer`, `Management`, `End-User`) |

## 1️⃣ Environment Setup & MDAAR Installation
Mount Google Drive for persistent storage and install **MDAAR**, bundled skills, and `cloudflared`.

In [ ]:
#@title 1. Initialize Runtime, Mount Drive & Install MDAAR { display-mode: "form" }
INSTALL_LOCAL_LLM_DEPS = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "True", "False"]

import os, sys, json, shutil, subprocess
from pathlib import Path

def _colab_or_env_bool(form_val, env_name, default=True):
    txt = str(form_val or '').strip()
    if txt and not txt.upper().startswith(('AUTO', 'FROM_ENV', 'FROM_SECRET')):
        return txt.lower() in ('1', 'true', 'yes', 'on')
    try:
        from google.colab import userdata
        sec = userdata.get(env_name)
        if sec is not None and str(sec).strip():
            return str(sec).strip().lower() in ('1', 'true', 'yes', 'on')
    except Exception:
        pass
    env_v = os.environ.get(env_name, '').strip()
    if env_v:
        return env_v.lower() in ('1', 'true', 'yes', 'on')
    return default

DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ Google Drive mounted successfully.')
    except Exception as e:
        print('⚠️ Drive mount notice (using local storage fallback):', e)

print('⏳ Installing MDAAR & dependencies...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)

install_local_deps = _colab_or_env_bool(INSTALL_LOCAL_LLM_DEPS, 'MDAAR_INSTALL_LOCAL_LLM_DEPS', default=True)
if install_local_deps:
    print('⏳ Installing local GPU quantization tools (transformers, accelerate, bitsandbytes)...')
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers', 'accelerate', 'bitsandbytes'], check=True)
else:
    print('⏩ Skipping local GPU libraries per MDAAR_INSTALL_LOCAL_LLM_DEPS=False (fast API mode).')

cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ Installing cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, SkillLoader
skills_count = len(SkillLoader().list_bundled_skills())
print(f'🎉 MDAAR v{__version__} ready with {skills_count} bundled guardrail skills!')

## 2️⃣ Source Repository Configuration (Git)
Leave fields blank / `AUTO` to load from Colab Secrets (`MDAAR_PROJECT_NAME`, `MDAAR_GIT_SOURCE_URL`, `MDAAR_SOURCE_AUTH`, `MDAAR_BASE_WORKSPACE`).

In [ ]:
#@title 2. Configure Source Repository & Test Connection { display-mode: "form" }
PROJECT_NAME = "" #@param {type:"string"}
GIT_SOURCE_URL = "" #@param {type:"string"}
SOURCE_AUTH_METHOD = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "HTTPS", "SSH"]
BASE_WORKSPACE_INPUT = "" #@param {type:"string"}

from pathlib import Path
from mdaar_pipeline import (
    MDAARPipeline,
    ModelGateway,
    DualModelManager,
    SecretSanitizer,
    resolve_setting,
)

resolved_project, src_proj = resolve_setting(PROJECT_NAME, ('MDAAR_PROJECT_NAME',), default='MySystem', return_source=True)
resolved_source_url, src_url = resolve_setting(GIT_SOURCE_URL, ('MDAAR_GIT_SOURCE_URL',), default='https://github.com/octocat/Hello-World.git', return_source=True)
resolved_source_auth, src_auth = resolve_setting(SOURCE_AUTH_METHOD, ('MDAAR_SOURCE_AUTH',), default='HTTPS', return_source=True)

default_ws = '/content/drive/MyDrive/AI_Documentation_Workspace' if os.path.exists('/content/drive/MyDrive') else '/content/AI_Documentation_Workspace'
resolved_ws, src_ws = resolve_setting(BASE_WORKSPACE_INPUT, ('MDAAR_BASE_WORKSPACE',), default=default_ws, return_source=True)
BASE_WORKSPACE = Path(resolved_ws)

print(f'📌 Project Name: {resolved_project} (source: {src_proj})')
print(f'🔗 Source URL: {SecretSanitizer.sanitize(resolved_source_url)} (source: {src_url})')
print(f'🔐 Auth Method: {resolved_source_auth} (source: {src_auth})')
print(f'📂 Workspace: {BASE_WORKSPACE} (source: {src_ws})')

pipeline = MDAARPipeline(
    project_name=resolved_project,
    base_workspace=BASE_WORKSPACE,
    git_source_url=resolved_source_url,
    source_auth=resolved_source_auth
)

if resolved_source_auth == "SSH":
    source_pub_key = pipeline.setup_source_ssh()
    print('='*70)
    print('🔑 Source Repository SSH Public Key (Deploy Key):')
    print(source_pub_key)
    print('='*70)

print('⏳ Testing connection to Source Repository...')
source_conn = pipeline.test_source_connection()
print(source_conn['message'])
if not source_conn['success'] and resolved_source_auth == "SSH":
    print('💡 Add the Deploy Key above in Settings -> Deploy Keys, then re-run this cell to verify.')

print(f'✅ Project workspace ready at: {pipeline.project_dir}')
print(f'🔍 Hybrid RAG Index path: {pipeline.rag_index_dir}')

## 3️⃣ External Shared Documentation Repository (Optional)
Automatically resolves `MDAAR_ENABLE_EXTERNAL_DOCS_REPO`, `MDAAR_GIT_DOCS_URL`, `MDAAR_DOCS_AUTH` when left empty/`AUTO`.

In [ ]:
#@title 3. Configure External Documentation Repository { display-mode: "form" }
ENABLE_EXTERNAL_DOCS_REPO = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "True", "False"]
GIT_DOCS_URL = "" #@param {type:"string"}
DOCS_AUTH_METHOD = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "HTTPS", "SSH"]

from mdaar_pipeline import SecretSanitizer, resolve_setting

enable_docs, src_en_docs = resolve_setting(ENABLE_EXTERNAL_DOCS_REPO, ('MDAAR_ENABLE_EXTERNAL_DOCS_REPO',), default=False, is_bool=True, return_source=True)
docs_url, src_docs_url = resolve_setting(GIT_DOCS_URL, ('MDAAR_GIT_DOCS_URL',), default='', return_source=True)
docs_auth, src_docs_auth = resolve_setting(DOCS_AUTH_METHOD, ('MDAAR_DOCS_AUTH',), default='HTTPS', return_source=True)

if enable_docs and docs_url.strip():
    pipeline.git_docs_url = docs_url.strip()
    pipeline.docs_auth = docs_auth
    print(f'🔗 Docs URL: {SecretSanitizer.sanitize(pipeline.git_docs_url)} (source: {src_docs_url}) | Auth: {docs_auth} ({src_docs_auth})')
    if docs_auth == "SSH":
        docs_pub_key = pipeline.setup_docs_ssh()
        print('='*70)
        print('🔑 Docs Repository SSH Public Key (Deploy Key with Write access):')
        print(docs_pub_key)
        print('='*70)
    print('⏳ Testing connection to External Documentation Repository...')
    docs_conn = pipeline.test_docs_connection()
    print(docs_conn['message'])
    if not docs_conn['success'] and docs_auth == "SSH":
        print('💡 Add the Deploy Key above with Write permission, then re-run this cell to confirm.')
else:
    pipeline.git_docs_url = None
    print(f'ℹ️ Docs and .rag_index/ will be saved locally (external enabled={enable_docs}, source={src_en_docs}).')

## 4️⃣ Dual-Model AI Architecture (`Planner Model Settings` + `Executor Model Settings`)
Choose between three providers for Planner (**4a**) and Executor (**4b**):
1. **`Gemini`:** Direct Google AI Studio access. Choose `auto-free-chain` to automatically cascade across active Free Tier models (`gemini-3.8-flash`, `gemini-3.6-flash`, `gemini-3.5-flash-lite`, `gemini-3.1-flash-lite`, `gemini-3-flash-preview`, `gemma-4-31b-it`, `gemma-4-26b-a4b-it`) with key rotation.
2. **`OpenAI-Gateway`:** Connect to **OpenRouter** (`https://openrouter.ai/api/v1` + `openrouter/free` or BYOK keys), **Cloudflare AI Gateway**, **Portkey**, **Groq**, **One-API**, **FreeLLM**, or **your 2nd-Colab BYOK Free Router (`free_router_colab.ipynb`)** using its Cloudflare tunnel URL + Bearer token + model `auto`.
3. **`Local-Offline`:** Quantized 4-bit HuggingFace inference on Colab T4 GPU.

In [ ]:
#@title 4a. Planner Model Settings (Master Blueprint & QA Review) { display-mode: "form" }
PLANNER_PROVIDER = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "Gemini", "OpenAI-Gateway", "Local-Offline"]
PLANNER_GEMINI_MODEL = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "auto-free-chain", "gemini-3.8-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemma-4-31b-it", "gemma-4-26b-a4b-it"]
PLANNER_GEMINI_KEYS_INPUT = "" #@param {type:"string"}
PLANNER_OPENAI_BASE_URL = "" #@param {type:"string"}
PLANNER_OPENAI_KEYS_INPUT = "" #@param {type:"string"}
PLANNER_OPENAI_MODELS = "" #@param {type:"string"}
PLANNER_LOCAL_MODEL_NAME = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
PLANNER_QUANTIZATION = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "4bit", "None"]
PLANNER_HF_TOKEN_INPUT = "" #@param {type:"string"}

from mdaar_pipeline import (
    DEFAULT_OPENAI_BASE_URL,
    DEFAULT_OPENAI_MODEL,
    DualModelManager,
    ModelGateway,
    resolve_setting,
)

def _build_planner_gw():
    _planner_provider, _src_prov = resolve_setting(PLANNER_PROVIDER, ('MDAAR_PLANNER_PROVIDER',), default='Gemini', return_source=True)
    _planner_gem_model, _src_gem_m = resolve_setting(PLANNER_GEMINI_MODEL, ('MDAAR_PLANNER_GEMINI_MODEL',), default='auto-free-chain', return_source=True)
    _planner_gem_keys_raw, _src_gem_k = resolve_setting(PLANNER_GEMINI_KEYS_INPUT, ('MDAAR_PLANNER_GEMINI_KEYS', 'GEMINI_API_KEY'), default='', return_source=True)
    _planner_oai_url, _src_oai_u = resolve_setting(PLANNER_OPENAI_BASE_URL, ('MDAAR_PLANNER_OPENAI_BASE_URL', 'MDAAR_ROUTER_URL', 'OPENAI_BASE_URL'), default=DEFAULT_OPENAI_BASE_URL, return_source=True)
    _planner_oai_keys_raw, _src_oai_k = resolve_setting(PLANNER_OPENAI_KEYS_INPUT, ('MDAAR_PLANNER_OPENAI_API_KEY', 'MDAAR_ROUTER_TOKEN', 'OPENROUTER_API_KEY', 'OPENAI_API_KEY'), default='', return_source=True)
    _planner_default_oai_model = DEFAULT_OPENAI_MODEL if 'openrouter.ai' in _planner_oai_url.lower() else ('llama-3.3-70b-versatile' if 'groq.com' in _planner_oai_url.lower() else 'auto')
    _planner_oai_models, _src_oai_m = resolve_setting(PLANNER_OPENAI_MODELS, ('MDAAR_PLANNER_OPENAI_MODELS', 'OPENAI_MODEL'), default=_planner_default_oai_model, return_source=True)
    _planner_local_model, _src_loc = resolve_setting(PLANNER_LOCAL_MODEL_NAME, ('MDAAR_PLANNER_LOCAL_MODEL',), default='Qwen/Qwen2.5-Coder-7B-Instruct', return_source=True)
    _planner_quant, _src_q = resolve_setting(PLANNER_QUANTIZATION, ('MDAAR_PLANNER_QUANTIZATION',), default='4bit', return_source=True)
    _planner_hf, _src_hf = resolve_setting(PLANNER_HF_TOKEN_INPUT, ('MDAAR_PLANNER_HF_TOKEN', 'HF_TOKEN'), default='', return_source=True)

    _planner_api_keys = _planner_oai_keys_raw if _planner_provider == 'OpenAI-Gateway' else _planner_gem_keys_raw
    _planner_model_name = _planner_oai_models if _planner_provider == 'OpenAI-Gateway' else _planner_gem_model
    _planner_gw = ModelGateway(
        provider=_planner_provider,
        model_name=_planner_model_name,
        api_keys=_planner_api_keys,
        base_url=_planner_oai_url,
        local_model_path=_planner_local_model,
        hf_token=_planner_hf,
        quantization=_planner_quant,
    )
    _k_count = len(_planner_gw.api_keys)
    _k_src = _src_oai_k if _planner_provider == 'OpenAI-Gateway' else _src_gem_k
    print(f'✅ Planner Model configured: {_planner_gw.descriptor}')
    print(f'   • Provider: {_planner_provider} ({_src_prov}) | Keys loaded: {_k_count} ({_k_src})')
    if _planner_provider == 'OpenAI-Gateway':
        print(f'   • Gateway URL: {_planner_oai_url} ({_src_oai_u}) | Model(s): {_planner_model_name} ({_src_oai_m})')
    elif _planner_provider == 'Gemini':
        print(f'   • Gemini Model/Chain: {_planner_gem_model} ({_src_gem_m})')
    return _planner_gw

planner_gateway = _build_planner_gw()
pipeline.planner_gateway = planner_gateway
pipeline.model_manager = DualModelManager(pipeline.planner_gateway, pipeline.executor_gateway)

In [ ]:
#@title 4b. Executor Model Settings (Full-Codebase Inspection & Doc Authoring) { display-mode: "form" }
EXECUTOR_MODE = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "Same as Planner Model", "Custom Executor Model"]
EXECUTOR_PROVIDER = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "Local-Offline", "Gemini", "OpenAI-Gateway"]
EXECUTOR_GEMINI_MODEL = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "auto-free-chain", "gemini-3.8-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemma-4-31b-it", "gemma-4-26b-a4b-it"]
EXECUTOR_GEMINI_KEYS_INPUT = "" #@param {type:"string"}
EXECUTOR_OPENAI_BASE_URL = "" #@param {type:"string"}
EXECUTOR_OPENAI_KEYS_INPUT = "" #@param {type:"string"}
EXECUTOR_OPENAI_MODELS = "" #@param {type:"string"}
EXECUTOR_LOCAL_MODEL_NAME = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
EXECUTOR_QUANTIZATION = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "4bit", "None"]
EXECUTOR_HF_TOKEN_INPUT = "" #@param {type:"string"}

from mdaar_pipeline import (
    DEFAULT_OPENAI_BASE_URL,
    DEFAULT_OPENAI_MODEL,
    DualModelManager,
    ModelGateway,
    resolve_setting,
)

resolved_exec_mode, src_exec_mode = resolve_setting(EXECUTOR_MODE, ('MDAAR_EXECUTOR_MODE',), default='same', return_source=True)
_use_same_as_planner = resolved_exec_mode.strip().lower() in ('same', 'single', 'planner', 'same as planner model', 'نفس مودل الخطة (مودل واحد)', '')

if _use_same_as_planner:
    executor_gateway = pipeline.planner_gateway
    print(f'ℹ️ Single-model mode active ({src_exec_mode}): Executor = Planner ({executor_gateway.descriptor}).')
else:
    def _build_executor_gw():
        _executor_provider, _src_prov = resolve_setting(EXECUTOR_PROVIDER, ('MDAAR_EXECUTOR_PROVIDER',), default='Local-Offline', return_source=True)
        _executor_gem_model, _src_gem_m = resolve_setting(EXECUTOR_GEMINI_MODEL, ('MDAAR_EXECUTOR_GEMINI_MODEL',), default='auto-free-chain', return_source=True)
        _executor_gem_keys_raw, _src_gem_k = resolve_setting(EXECUTOR_GEMINI_KEYS_INPUT, ('MDAAR_EXECUTOR_GEMINI_KEYS', 'MDAAR_PLANNER_GEMINI_KEYS', 'GEMINI_API_KEY'), default='', return_source=True)
        _executor_oai_url, _src_oai_u = resolve_setting(EXECUTOR_OPENAI_BASE_URL, ('MDAAR_EXECUTOR_OPENAI_BASE_URL', 'MDAAR_ROUTER_URL', 'MDAAR_PLANNER_OPENAI_BASE_URL', 'OPENAI_BASE_URL'), default=DEFAULT_OPENAI_BASE_URL, return_source=True)
        _executor_oai_keys_raw, _src_oai_k = resolve_setting(EXECUTOR_OPENAI_KEYS_INPUT, ('MDAAR_EXECUTOR_OPENAI_API_KEY', 'MDAAR_ROUTER_TOKEN', 'MDAAR_PLANNER_OPENAI_API_KEY', 'OPENROUTER_API_KEY', 'OPENAI_API_KEY'), default='', return_source=True)
        _executor_default_oai_model = DEFAULT_OPENAI_MODEL if 'openrouter.ai' in _executor_oai_url.lower() else ('llama-3.3-70b-versatile' if 'groq.com' in _executor_oai_url.lower() else 'auto')
        _executor_oai_models, _src_oai_m = resolve_setting(EXECUTOR_OPENAI_MODELS, ('MDAAR_EXECUTOR_OPENAI_MODELS', 'MDAAR_PLANNER_OPENAI_MODELS', 'OPENAI_MODEL'), default=_executor_default_oai_model, return_source=True)
        _executor_local_model, _src_loc = resolve_setting(EXECUTOR_LOCAL_MODEL_NAME, ('MDAAR_EXECUTOR_LOCAL_MODEL',), default='Qwen/Qwen2.5-Coder-7B-Instruct', return_source=True)
        _executor_quant, _src_q = resolve_setting(EXECUTOR_QUANTIZATION, ('MDAAR_EXECUTOR_QUANTIZATION',), default='4bit', return_source=True)
        _executor_hf, _src_hf = resolve_setting(EXECUTOR_HF_TOKEN_INPUT, ('MDAAR_EXECUTOR_HF_TOKEN', 'MDAAR_PLANNER_HF_TOKEN', 'HF_TOKEN'), default='', return_source=True)

        _executor_api_keys = _executor_oai_keys_raw if _executor_provider == 'OpenAI-Gateway' else _executor_gem_keys_raw
        _executor_model_name = _executor_oai_models if _executor_provider == 'OpenAI-Gateway' else _executor_gem_model
        _executor_gw = ModelGateway(
            provider=_executor_provider,
            model_name=_executor_model_name,
            api_keys=_executor_api_keys,
            base_url=_executor_oai_url,
            local_model_path=_executor_local_model,
            hf_token=_executor_hf,
            quantization=_executor_quant,
        )
        _k_count = len(_executor_gw.api_keys)
        _k_src = _src_oai_k if _executor_provider == 'OpenAI-Gateway' else _src_gem_k
        print(f'✅ Custom Executor Model configured: {_executor_gw.descriptor}')
        print(f'   • Provider: {_executor_provider} ({_src_prov}) | Keys loaded: {_k_count} ({_k_src})')
        return _executor_gw
    executor_gateway = _build_executor_gw()

pipeline.executor_gateway = executor_gateway
pipeline.gateway = executor_gateway
pipeline.model_manager = DualModelManager(pipeline.planner_gateway, pipeline.executor_gateway)
print(f'🧠 Planner: {pipeline.planner_gateway.descriptor}  |  ⚙️ Executor: {pipeline.executor_gateway.descriptor}')

## 5️⃣ Fail-Fast Code Synchronization & Diff Tracker
Clones/pulls source code and aborts immediately if clone fails or yields an empty folder (preventing hallucinated docs).

In [ ]:
#@title 5. Sync Code and Detect Changes { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 Current Git Commit: ({current_commit[:7] if current_commit else "Local"})')
if changed_files:
    print(f'🔍 Detected {len(changed_files)} changed files since last documented commit:')
    for f in changed_files[:15]:
        print('  •', f)
else:
    print('ℹ️ Full fresh repository inspection will run across all tracked source files.')

## 6️⃣ 100% Full-File Map-Reduce Generation, Quality Gate & RAG Indexing
Runs fresh full-repo extraction, Executor Map pass across 100% of files, Planner Master Blueprint, full-package authoring, Quality Gate (`L0`/`L1`), and persists `.rag_index/`.

In [ ]:
#@title 6. Execute 100% Full-Codebase Scan & Multi-Agent Generation { display-mode: "form" }
FORCE_FULL_GENERATION = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "True", "False"]

from mdaar_pipeline import resolve_setting

force_full, src_ff = resolve_setting(FORCE_FULL_GENERATION, ('MDAAR_FORCE_FULL',), default=True, is_bool=True, return_source=True)
print(f'⚙️ Full 100% Scan Mode: {force_full} (source: {src_ff})')

def _show_map_progress(idx, total, module_name, files_desc):
    if idx == 1 or idx == total or idx % 5 == 0:
        print(f'  [{idx}/{total}] Inspecting `{module_name}`: {files_desc[:90]}')

target_changes = None if force_full else changed_files
run_stats = pipeline.run_multi_agent_generation(
    changed_files=target_changes,
    force_full=force_full,
    progress_callback=_show_map_progress
)

if current_commit:
    pipeline.save_state(current_commit)

print('='*70)
print('📊 Documentation Pipeline Telemetry:')
print(json.dumps(run_stats, indent=2, ensure_ascii=False))
print('='*70)

## 7️⃣ Documentation Portal & Cloudflare Tunnel
Build and launch interactive MkDocs Material site and push docs hub updates if configured.

In [ ]:
#@title 7. Launch Portal and Push to Remote Docs Repo { display-mode: "form" }
DOCS_PORT = "" #@param {type:"string"}

from mdaar_pipeline import resolve_setting

port_str, src_port = resolve_setting(DOCS_PORT, ('MDAAR_DOCS_PORT',), default='8000', return_source=True)
docs_port = int(port_str or 8000)
tunnel_url = pipeline.serve_and_tunnel(port=docs_port)

if tunnel_url:
    print('='*70)
    print(f'🎉 Interactive Documentation Portal is live on port {docs_port} ({src_port}):')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ Server running locally, check tunnel logs.')

sync_docs_res = pipeline.sync_docs_repo()
print('📡 Remote Docs Sync:', sync_docs_res)

## 8️⃣ Hybrid BM25 Q&A Assistant (`.rag_index/`)
Query docs + codebase FileFactCards + DB schemas + API catalogs stored in `project_dir/.rag_index/`.

In [ ]:
#@title 8. Query Project Documentation & Verified Codebase Facts { display-mode: "form" }
QUESTION = "" #@param {type:"string"}
AUDIENCE = "AUTO (from Colab Secrets/Env)" #@param ["AUTO (from Colab Secrets/Env)", "Developer", "Management", "End-User"]

from mdaar_pipeline import resolve_setting

resolved_q = resolve_setting(QUESTION, ('MDAAR_RAG_QUESTION',), default='What is the system architecture, database schema, and core APIs?')
resolved_aud = resolve_setting(AUDIENCE, ('MDAAR_RAG_AUDIENCE',), default='Developer')

chat_engine = pipeline.get_chat_engine()
chunks_loaded = chat_engine.load_documents(prefer_saved_index=True)
print(f'📚 Loaded {chunks_loaded} indexed knowledge chunks from `{chat_engine.index_dir}`')
print(f'🔍 Querying for: "{resolved_q}" (Audience: {resolved_aud})...\n')

answer = chat_engine.ask(resolved_q, audience=resolved_aud)
print('='*70)
print(answer)
print('='*70)